# FeCAM CIFAR-100 reproduction on Kaggle

This notebook runs the repository's full CIFAR-100 B=50, increment=10 NCM and FeCAM YAML configurations. Each classifier is a separate 200-epoch run. NCM uses Euclidean distance with L2-normalized test features and prototypes (`normalize_features: true`, `normalize_prototypes: true`), matching the authors' released implementation.

## Before running
1. Push/merge the updated reproduction YAML and runner to the public GitHub `dev` branch. Kaggle clones `dev`, so local or unpushed changes will not be included.
2. Upload this notebook to Kaggle, turn **Internet** on, and select **GPU T4 x2** in Notebook Settings.
3. Choose **Save Version → Save & Run All**. The run cell launches NCM on `cuda:0` and FeCAM on `cuda:1` concurrently, then writes summaries, timings, a CSV, logs, and a ZIP under `/kaggle/working`.

This notebook writes to versioned `kaggle-reproduction-v2-*` folders so it won't mistake results from the earlier raw-NCM configuration for the new run. A full run may take several hours. The runner saves an atomic training checkpoint after every epoch, including optimizer, scheduler, and random-number states. Re-running in the same Kaggle working directory resumes from those checkpoints. If Kaggle starts a fresh working directory, attach a `fecam-resume` dataset with checkpoints trained under the exact current config; incompatible checkpoints will be rejected. Each classifier still uses one GPU.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/duongthienphu/fecam-study.git'
REPO_BRANCH = 'feat/evaluation-tracking'
REPO_DIR = Path('/kaggle/working/fecam-study')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, 'FETCH_HEAD'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)
print('Branch:', subprocess.run(['git', 'branch', '--show-current'], cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout.strip())
print('Commit:')
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO_DIR, check=True)

In [ ]:
# Install only data/config dependencies. Keep Kaggle's CUDA-enabled PyTorch intact.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets==4.8.5', 'PyYAML==6.0.3'], check=True)
import torch
import torchvision
import datasets
import yaml

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. In Kaggle Settings, select GPU, then restart the session.')
print('GPU count:', torch.cuda.device_count())
if torch.cuda.device_count() < 2:
    raise RuntimeError('Select GPU T4 x2 in Kaggle Settings; this notebook assigns one classifier to each GPU.')
print('NCM GPU:', torch.cuda.get_device_name(0))
print('FeCAM GPU:', torch.cuda.get_device_name(1))

In [ ]:
import csv
import json
import shutil
import time
from datetime import datetime, timezone

OUTPUT_DIR = REPO_DIR / 'outputs'
RESULTS_CSV = Path('/kaggle/working/fecam_reproduction_results.csv')
RESULTS_ZIP = Path('/kaggle/working/fecam_reproduction_results.zip')
CSV_FIELDS = [
    'classifier', 'protocol', 'seed', 'epochs', 'device', 'assigned_gpu',
    'avg_acc_percent', 'final_acc_percent', 'bwt_percent',
    'dataset_prepare_seconds', 'base_training_epoch_seconds', 'base_training_wall_seconds',
    'last_session_wall_clock_seconds', 'last_session_wall_clock_minutes', 'summary_path',
]

def refresh_result_files():
    summaries = sorted(OUTPUT_DIR.glob('kaggle-reproduction-v2-*/summary.json'))
    rows = []
    for summary_path in summaries:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        ev, metrics = summary['evaluation'], summary['evaluation']['metrics']
        rows.append({
            'classifier': ev['classifier']['name'],
            'protocol': summary['config']['protocol'],
            'seed': summary['config']['seed'],
            'epochs': summary['config']['training']['epochs'],
            'device': summary['device'],
            'assigned_gpu': summary['notebook_assigned_gpu'],
            'avg_acc_percent': metrics['avg_acc'],
            'final_acc_percent': metrics['last_acc'],
            'bwt_percent': metrics['bwt'],
            'dataset_prepare_seconds': summary['notebook_dataset_prepare_seconds'],
            'base_training_epoch_seconds': round(sum(epoch['seconds'] for epoch in summary['base_training']), 3),
            'base_training_wall_seconds': summary['base_training_wall_seconds'],
            'last_session_wall_clock_seconds': summary['notebook_wall_clock_seconds'],
            'last_session_wall_clock_minutes': round(summary['notebook_wall_clock_seconds'] / 60, 3),
            'summary_path': str(summary_path),
        })
    with RESULTS_CSV.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)
    shutil.make_archive(str(RESULTS_ZIP.with_suffix('')), 'zip', root_dir=REPO_DIR, base_dir='outputs')
    return rows

def run_both_parallel():
    if torch.cuda.device_count() < 2:
        raise RuntimeError('Two GPUs are required: select GPU T4 x2 in Kaggle Settings.')
    from datasets import load_dataset
    prep_start = time.perf_counter()
    data = load_dataset('uoft-cs/cifar100', revision='aadb3af77e9048adbea6b47c21a81e47dd092ae5',
                        cache_dir=str(REPO_DIR / 'data' / 'huggingface'))
    dataset_prepare_seconds = time.perf_counter() - prep_start
    print('Pinned CIFAR-100 ready:', {name: len(split) for name, split in data.items()})
    print(f'Dataset download/cache preparation: {dataset_prepare_seconds:.1f}s')
    del data
    import gc
    gc.collect()

    child_env = os.environ.copy()
    child_env['OMP_NUM_THREADS'] = '2'
    child_env['MKL_NUM_THREADS'] = '2'
    jobs = {}
    resume_input = Path('/kaggle/input/fecam-resume')
    for classifier, gpu_id in [('ncm', 0), ('fecam', 1)]:
        run_name = f'kaggle-reproduction-v2-{classifier}-l2norm' if classifier == 'ncm' else f'kaggle-reproduction-v2-{classifier}'
        output = OUTPUT_DIR / run_name
        checkpoint_path = output / 'base_training_checkpoint.pt'
        summary_path = output / 'summary.json'
        input_checkpoint = resume_input / classifier / 'base_training_checkpoint.pt'
        if not checkpoint_path.exists() and input_checkpoint.is_file():
            output.mkdir(parents=True, exist_ok=True)
            shutil.copy2(input_checkpoint, checkpoint_path)
        if summary_path.exists():
            print(f'{classifier.upper()} already completed; keeping {summary_path}')
            continue
        config = REPO_DIR / 'configs' / 'reproduction' / f'cifar100_{classifier}.yaml'
        config_data = yaml.safe_load(config.read_text(encoding='utf-8'))
        classifier_config = config_data['classifier']
        if classifier == 'ncm' and (
            not classifier_config.get('normalize_features', False)
            or not classifier_config.get('normalize_prototypes', False)
            or classifier_config.get('metric') != 'euclidean'
        ):
            raise ValueError('NCM config must use Euclidean distance with L2-normalized features and prototypes.')
        print(f'{classifier.upper()} config: {classifier_config}')
        command = [sys.executable, 'scripts/run_experiment.py', '--config', str(config),
                   '--device', f'cuda:{gpu_id}', '--output', str(output)]
        if output.exists():
            if not checkpoint_path.is_file():
                raise FileNotFoundError(f'Output exists without a resumable checkpoint: {output}')
            command.append('--resume')
            print(f'Resuming {classifier.upper()} from epoch checkpoint.')
        log_path = Path('/kaggle/working') / f'{run_name}.log'
        log_handle = log_path.open('w', encoding='utf-8')
        process = subprocess.Popen(command, cwd=REPO_DIR, env=child_env,
                                   stdout=log_handle, stderr=subprocess.STDOUT, text=True)
        jobs[classifier] = {'process': process, 'gpu': gpu_id, 'output': output,
                           'log_path': log_path, 'log_handle': log_handle,
                           'start': time.perf_counter()}
        print(f'Started {classifier.upper()} on cuda:{gpu_id}; log: {log_path}')

    while any(job['process'].poll() is None for job in jobs.values()):
        time.sleep(60)
        status = []
        for name, job in jobs.items():
            elapsed = time.perf_counter() - job['start']
            state = 'running' if job['process'].poll() is None else 'finished'
            status.append(f'{name.upper()} cuda:{job["gpu"]} {state}, {elapsed / 60:.1f} min')
        print(' | '.join(status), flush=True)

    failures = []
    for classifier, job in jobs.items():
        return_code = job['process'].wait()
        job['log_handle'].close()
        elapsed = time.perf_counter() - job['start']
        job['output'].mkdir(parents=True, exist_ok=True)
        shutil.copy2(job['log_path'], job['output'] / 'runner.log')
        summary_path = job['output'] / 'summary.json'
        if return_code != 0:
            failures.append(classifier)
            print(f'{classifier.upper()} failed; log tail:\n{job["log_path"].read_text(encoding="utf-8", errors="replace")[-10000:]}')
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        summary['notebook_wall_clock_seconds'] = round(elapsed, 3)
        summary['notebook_dataset_prepare_seconds'] = round(dataset_prepare_seconds, 3)
        summary['notebook_assigned_gpu'] = f'cuda:{job["gpu"]}'
        summary['notebook_wall_clock_includes'] = 'elapsed time in the most recent Kaggle session; for resumed runs, sum base_training seconds for total backbone training time'
        summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
        print(f'{classifier.upper()} on cuda:{job["gpu"]}: {elapsed:.1f}s ({elapsed / 60:.2f} min)')

    rows = refresh_result_files()
    print('Comparison CSV:', RESULTS_CSV)
    print('Download ZIP:', RESULTS_ZIP)
    if failures:
        raise RuntimeError('Failed classifier run(s): ' + ', '.join(failures))
    return rows

## Run both classifiers on separate GPUs

This cell preloads the pinned dataset once, then starts the 200-epoch NCM and FeCAM YAMLs concurrently on `cuda:0` and `cuda:1`. It prints each loaded classifier configuration and checks that NCM uses Euclidean distance with both L2-normalization flags enabled. It limits each process to two CPU math threads to reduce contention on Kaggle's four CPU cores.


In [ ]:
comparison_rows = run_both_parallel()
comparison_rows

## Downloadable results

After both runs, this cell displays the comparison CSV and ZIP. In Kaggle, use the notebook Output panel or save a notebook version to download files from `/kaggle/working`.

In [ ]:
from IPython.display import display, FileLink
rows = refresh_result_files()
display(rows)
display(FileLink(str(RESULTS_CSV)))
display(FileLink(str(RESULTS_ZIP)))